## Import the libraries

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import datetime
from scipy.optimize import fsolve, root
import math
import matplotlib.pyplot as plt

# Check if coordinate is land ?
import shapely.geometry as sgeom
import cartopy.io.shapereader as shpreader
from shapely.ops import unary_union
from shapely.prepared import prep


# Define is_land function : 

# Custom landmap (removing some islands) : 
land_shp_fname = '50m_land_edited.shp'

land_geom = unary_union(list(shpreader.Reader(land_shp_fname).geometries()))
land = prep(land_geom)
def is_land(long, lat):
    """
    Function to assert whether a longitude-latitude pair is overland or at sea.
    """
    return land.contains(sgeom.Point(long, lat))


## Load Hurdat2 dataset

In [3]:
ed01 = pd.read_csv('edited_atlantic_storms.csv')
display(ed01.head(5))

## Load Sea Surface Temperature values

In [4]:
# Access a SST value based on grid id or based on lat-long current values
hadsst = xr.open_dataset('sliced_HadISST.nc')
display(hadsst)

In [5]:
hadsst_pre1870=xr.open_dataset('sst_pre_1870.nc')
hadsst_post2020=xr.open_dataset('sst_post_2020.nc')
# Quick sanity check : This should output a file with only 12 monthly values available (coordinates variable)
display(hadsst_pre1870)

## Load Stratosphere temperature values

In [6]:
air_temp = xr.open_dataset('air_mon_temp_sliced.nc')
air_temp

In [7]:
air_temp_post_2015 = xr.open_dataset('air_mon_temp_sliced_2015-2021.nc')
air_temp_post_2015

## Modify the dataframe

In [8]:
datasets = ed01[['latitude','longitude','date']].copy()
datasets['date'] = pd.to_datetime(datasets['date'], format ='%Y-%m-%d %H:%M:%S')
display(datasets.head(5))

In [9]:
if 'SST' in ed01.columns : 
    print('The Sea Surface temperature values already exist. No calculations required!')
    sst_values = ed01['SST']
    datasets['SST'] = sst_values
else : 
    sst_values = []
    
    for value in datasets.iterrows() :
        date = value[1][2]
        lat_i = value[1][0]
        long_i = value[1][1]
        land_realm = is_land(long_i,lat_i) # False = ocean, True = Land 
            
        if land_realm == True : 
            sst_values.append(np.nan)
        else : 
            if date.year < 1870 :
                temperature = hadsst_pre1870.interp(latitude=lat_i,longitude =long_i,month=date.month).sst.item()
                if np.isnan(temperature) :
                    temperature = hadsst_pre1870.interp(latitude =lat_i, longitude =long_i, month= date.month, method = 'nearest').sst.item()

            # Check if the hurricane stems from a post-2020 dataset :                 
            elif (date.year == 2020) and (date.month > 1) :
                temperature = hadsst_post2020.interp(latitude=lat_i,longitude =long_i,month=date.month).sst.item()
                if np.isnan(temperature) :
                    temperature = hadsst_post2020.interp(latitude =lat_i, longitude =long_i, month= date.month, method = 'nearest').sst.item()
            # Else, just grab the actual value : 
            else :
                temperature = hadsst.interp(latitude =lat_i, longitude =long_i, time = date).sst.item()
                if np.isnan(temperature) :
                    temperature = hadsst.interp(latitude =lat_i, longitude =long_i, time=date, method = 'nearest').sst.item()
            sst_values.append(temperature)
    datasets['SST'] = sst_values
    ed01['SST'] = sst_values

In [10]:
if 'T0' in ed01.columns : 
    print('Air 100 mbar temperature values already exist. No calculations required!')
    T0_values = ed01['T0']
    datasets['T0'] = T0_values
else : 
    T0_values = []

    for value in datasets.iterrows() :
        date = value[1][2]
        lat_i = value[1][0]
        long_i = value[1][1]
        land_realm = is_land(long_i,lat_i) # False = ocean, True = Land 
        
        if land_realm == True : 
            T0_values.append(np.nan)
        else :
            if date.year > 2015 :
                temperature = air_temp_post_2015.interp(lat=lat_i,lon =long_i+360,time=date).air.item()

            # Else, just take the value in the Reanalysis V3 dataset : 
            else :
                temperature = air_temp.interp(lat =lat_i, lon =long_i+360, time = date).air.item()
            T0_values.append(temperature)
    datasets['T0'] = T0_values
    ed01['T0'] = T0_values

In [11]:
datasets['c_i'] = ed01['Vi']/1.852
datasets['Vmax'] = ed01['maximum_sustained_wind_knots']
display(datasets.head(5))

In [12]:
if 'pc_calc' in ed01.columns :  
    print('Central pressure values already available. No calculations required!')
    pc_calc = ed01['pc_calc']
else : 
    pc_calc = []

    for row in datasets.iterrows() :
        lat_i = row[1][0]
        c_i = row[1][4]
        vmax = row[1][5]

        Vg_calc = vmax - 1.5*c_i**0.63
        pc = 1013-((Vg_calc-5.843+0.558*lat_i)/14.118)**2
        pc_calc.append(pc)

In [13]:
datasets['pc'] = pc_calc
datasets['real_pc'] = ed01['maximum_pressure']
datasets['diff_pc'] = datasets['real_pc'] - datasets['pc']
display(datasets.head(5))

In [14]:
# Create a composite column which takes readings in, but fills missing values with calculated pressures : 
datasets['pc_comp'] = datasets['real_pc'].fillna(datasets['pc'])
datasets['pc_compv2'] = np.where(datasets['pc_comp']> 1013, 1013,datasets['pc_comp'])
display(datasets.head(5))

## Some validations and visualizations : 

In [15]:
print(f'pc value extremes :\n Max == {datasets["pc"].max()},\n Min == {datasets["pc"].min()}')
print(f'The differences :\n Max == {datasets["diff_pc"].max()},\n Min == {datasets["diff_pc"].min()}')

print(f'Some statistics on the errors :\n Mean == {datasets["diff_pc"].mean():.2f}'
      f',\n Standard deviation == {datasets["diff_pc"].std():.2f}')
# Plot the differences (seek the magnitude) : 
datasets.plot(y = 'diff_pc', lw =0.3)

In [16]:
# Store everything in the output dataframe : 
ed01['pc_calc'] = pc_calc
# real_pc is not stored to ed01, as "maximum pressure" is the original recording.
ed01['diff_pc'] = datasets['diff_pc']
ed01['pc_comp'] = datasets['pc_comp']
ed01['pc_final'] = datasets['pc_compv2']
display(ed01.head(5))

## Calculate intensities 

In [17]:
# Relative humidity (taken from Darling 1991) : 
RH = 0.8
# Gas constant of water vapor (taken from Darling 1991) : 
Rv = 461.5
# Temperature at top of troposphere (100 mb), taken as 203 Kelvin (according to Vickery 2000) : 
datasets['T0'] = datasets['T0']

In [18]:
#Carnot engine efficiency
datasets['eps'] = (datasets['SST']- datasets['T0'])/ datasets['SST']
#Latent heat of vaporization
datasets['Lv'] = 2.5*(10**6)-2320*(datasets['SST']-273)
#Staturation vapor pressure
datasets['e_s'] = 6.112*np.exp((17.67*(datasets['SST']-273))/(datasets['SST']-29.5))
#Surface value of partial pressure of ambiant dry air
datasets['pda'] = 1013 - (RH*datasets['e_s'])
#Parameter A 
datasets['A'] = (datasets['eps']*datasets['Lv']*datasets['e_s'])/((1-datasets['eps'])*Rv*datasets['SST']*datasets['pda'])
#Parameter B 
datasets['B'] = RH*(1+(datasets['e_s']*np.log(RH))/(datasets['pda']*datasets['A']))
display(datasets.head(5))

In [19]:
# Nonlinear equation to solve (taken from Darling 1991, solved as in Emmanuel 1988): 
def central_pressure_exp_eq(x,A,B) :
    return np.exp((-A/x+B*A))-x

In [20]:
# Use a first "guess" value as close to zero as possible :
x1 = np.ones((datasets.shape[0]))-0.9
# Use another first "guess" value, this time as reasonably high as possible : 
x2 = x1+10

# Don't let the solver quietly ignore values going too high : 
np.seterr(all= 'raise')

# Recorders : 
solutions = []
non_SST = 0
# Start solving : 
for i in range(datasets.shape[0]) :
    if np.isnan(datasets['A'][i]) | np.isnan(datasets['B'][i]) | np.isnan(datasets['T0'][i]) : 
        s = np.array([np.nan])
        non_SST+=1
    else :
        try : 
            s1 = fsolve(central_pressure_exp_eq,x0=x1[i],args = (datasets['A'][i],datasets['B'][i]))
            s2 = fsolve(central_pressure_exp_eq,x0=x2[i],args=(datasets['A'][i],datasets['B'][i]))

            
            # According to Emmanuel 1988, the first intercept is the only "stable" solution.
            s = min(s1,s2)
        except :
            # In some cases, division by zero will occur. Also, convergence might fail. To log this,
            #simply uncomment the print statement.
            #print('Warning raised.', datasets["A"][i],datasets["B"][i],i)
            
            s = fsolve(central_pressure_exp_eq,x0=0.7,args = (datasets['A'][i],datasets['B'][i]))
            if s == 0 : 
                s = np.array([np.nan])
            pass
    
    
    solutions.append(s)
solutions = np.ndarray.flatten(np.array(solutions))
datasets['x'] = solutions
print(f'The amount of datasets which could not be used is : {np.count_nonzero(np.isnan(solutions))}')
print(f'{non_SST} of the invalid datasets simply did not correspond to a SST value.')

datasets['pdc'] = datasets['x']*datasets['pda']

In [21]:
datasets['Int'] = (1013-datasets['pc_compv2']+(1-RH)*datasets['e_s'])/((1-datasets['x'])*(1013-(RH*datasets['e_s'])))
datasets['Int'] = np.where(datasets['Int']< 0,0,datasets['Int'])
datasets['ln_Int'] = np.log(datasets['Int'], where = (datasets['Int'] !=0))
print(f'Intensity value extremes :\n Max == {datasets["Int"].max()},\n Min == {datasets["Int"].min()}')

In [22]:
display(datasets.head(5))

In [23]:
# Plot the intensities.
# According to Darling 1991, values up to 1.26 may be expected. Hurricanes going beyond a latitude of 34 should be ignored.
datasets.reset_index().plot.scatter(x= 'index',y = 'Int', s =0.3)
plt.axhline(y=1.26, color='r', linestyle='-')
plt.axhline(y=1, color='orange', linestyle='-')

# Final formatting steps in preparation for regression analysis 

In [31]:
intensity_params_df = pd.DataFrame()

for group, value in grouped_ds :

    if value.shape[0] > 1 :
    
        intensities = np.array(value['Int'])
        ln_intensities = np.array(value['ln_Int'])

        # Adding information about one timestep earlier : 
        intensities_i_m1 = np.insert(intensities,0,np.nan)  # There are no intensity reading before the first timestep
        #Beware of a little quirk on "np.insert" -> the value is added "before" the given position, thus -1 is not working.        
        intensities_im1 = intensities_i_m1[:-1] # Pop the last value
        ln_int_im1 = np.log(intensities_im1, where = (intensities_im1 !=0))        
        value['Int_im1'] = intensities_im1
        value['ln_Int_im1'] = ln_int_im1
        
        # Adding information about two timesteps earlier (persistence)
        intensities_i_m2 = np.insert(intensities_im1,0,np.nan) # There are no intensity reading before the first timestep
        intensities_im2 = intensities_i_m2[:-1] # Pop the last value
        ln_int_im2 = np.log(intensities_im2, where = (intensities_im2 !=0)) 
        value['Int_im2'] = intensities_im2
        value['ln_Int_im2'] = ln_int_im2
        
        # Adding information on the next intensity        
        intensities_ip1 = np.insert(intensities[1:],len(intensities)-1,np.nan)
        ln_intensities_ip1 = np.insert(ln_intensities[1:],len(ln_intensities)-1,np.nan)        
        value['Int_ip1'] = intensities_ip1
        value['ln_Int_ip1'] = ln_intensities_ip1
        
        # Adding information on the next SST
        SSTs_p1 = np.array(value['SST'][1:]) # Skip the first value
        SSTs_p1 = np.insert(SSTs_p1,len(SSTs_p1),np.nan) # Last value does not exist
        value['SST_p1'] = SSTs_p1
        

        
    intensity_params_df = pd.concat([intensity_params_df,value])
display(intensity_params_df.head(5))        

## Saving everything

In [32]:
ed01['Int'] = datasets['Int']
ed01['ln_Int'] = datasets['ln_Int']
ed01['Int_im1'] = intensity_params_df['Int_im1']
ed01['ln_Int_im1'] = intensity_params_df['ln_Int_im1']
ed01['Int_im2'] = intensity_params_df['Int_im2']
ed01['ln_Int_im2'] = intensity_params_df['ln_Int_im2']
ed01['Int_ip1'] = intensity_params_df['Int_ip1']
ed01['ln_Int_ip1'] = intensity_params_df['ln_Int_ip1']
ed01['SST_p1'] = intensity_params_df['SST_p1']
ed01['delta_SST'] = ed01['SST_p1']-ed01['SST']
ed01.to_csv('edited_atlantic_storms.csv', index = False)
display(ed01)

In [33]:
storms_i = pd.read_csv('edited_atlantic_storms.csv').groupby('id', as_index = False).first()
display(storms_i)